# SQL Foundations — Databases, Tables & Keys

## Objective
Learn how a relational database is structured, and build one by hand:
- `CREATE TABLE` with data types and constraints
- `PRIMARY KEY` and `FOREIGN KEY`
- `INSERT INTO` (single row and multi-row)
- `PRAGMA foreign_keys = ON`
- Verifying data with `SELECT *`

We run real SQL from Python using the built-in `sqlite3` module, so every query below actually executes and shows real output.

## Imports

In [1]:
import sqlite3          # built-in Python library for working with SQLite databases
import pandas as pd     # used only to display query results as clean tables

# Connect to an in-memory SQLite database (lives only for this notebook run)
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Turn on Foreign Key enforcement — OFF by default in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")
print("Connected. Foreign key enforcement is ON.")

Connected. Foreign key enforcement is ON.


## Theory
A **relational database** splits data into linked tables instead of one repetitive flat table:

- **Primary Key (PK):** uniquely identifies each row in a table (e.g. `team_id`)
- **Foreign Key (FK):** a column that points to another table's Primary Key — this link is what makes it "relational"
- **Logical execution order:** `FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT` (how SQL actually runs, vs. how it's written)

## Example 1 — Creating the `teams` table

In [2]:
cursor.execute("""
CREATE TABLE teams (
    team_id INTEGER PRIMARY KEY,   -- unique ID for each team
    team_name TEXT NOT NULL,       -- team's name, can't be empty
    city TEXT,                     -- team's home city
    founded_year INTEGER           -- year the team was founded
);
""")
print("teams table created.")

teams table created.


## Example 2 — Creating `players` with a Foreign Key link to `teams`

In [3]:
cursor.execute("""
CREATE TABLE players (
    player_id INTEGER PRIMARY KEY,          -- unique ID for each player
    player_name TEXT NOT NULL,              -- player's name, required
    age INTEGER,                            -- player's age
    position TEXT,                          -- e.g. 'Forward', 'Goalkeeper'
    team_id INTEGER,                        -- link column -> points to a team
    FOREIGN KEY (team_id) REFERENCES teams(team_id)  -- enforce the link
);
""")
print("players table created.")

# Insert parent data (teams) BEFORE child data (players) -- Foreign Keys need the target row to exist first
cursor.executemany(
    "INSERT INTO teams (team_id, team_name, city) VALUES (?, ?, ?)",
    [(1, "FC Lahore Lions", "Lahore"),
     (2, "Karachi Kings FC", "Karachi")]
)

cursor.executemany(
    "INSERT INTO players (player_id, player_name, age, position, team_id) VALUES (?, ?, ?, ?, ?)",
    [(101, "Ali Raza", 24, "Forward", 1),
     (102, "Shafi", 19, "Winger", 1),
     (103, "Shafay", 18, "Left Back", 2)]
)
conn.commit()

# Verify with a SELECT, displayed as a clean pandas table
pd.read_sql_query("SELECT * FROM players;", conn)

players table created.


,player_id,player_name,age,position,team_id
0,101,Ali Raza,24,Forward,1
1,102,Shafi,19,Winger,1
2,103,Shafay,18,Left Back,2


## Practice Exercise

In [4]:
# Your Turn: create a `stadiums` table yourself
# Columns: stadium_id (PK), stadium_name (TEXT, required), city (TEXT), capacity (INTEGER)

cursor.execute("""
CREATE TABLE stadiums (
    stadium_id INTEGER PRIMARY KEY,
    stadium_name TEXT NOT NULL,
    city TEXT,
    capacity INTEGER
);
""")

cursor.executemany(
    "INSERT INTO stadiums (stadium_id, stadium_name, city, capacity) VALUES (?, ?, ?, ?)",
    [(1, "Lahore Arena", "Lahore", 35000),
     (2, "Multan Arena", "Multan", 25000),
     (3, "Jhang Arena", "Jhang", 10000)]
)
conn.commit()

pd.read_sql_query("SELECT * FROM stadiums;", conn)

,stadium_id,stadium_name,city,capacity
0,1,Lahore Arena,Lahore,35000
1,2,Multan Arena,Multan,25000
2,3,Jhang Arena,Jhang,10000


## Mini Challenge — link `matches` to teams AND stadiums

In [5]:
cursor.execute("""
CREATE TABLE matches (
    match_id INTEGER PRIMARY KEY,
    home_team_id INTEGER,
    away_team_id INTEGER,
    stadium_id INTEGER,
    FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (away_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (stadium_id) REFERENCES stadiums(stadium_id)
);
""")

cursor.executemany(
    "INSERT INTO matches (home_team_id, away_team_id, stadium_id) VALUES (?, ?, ?)",
    [(1, 2, 1)]   # FC Lahore Lions vs Karachi Kings FC, at Lahore Arena
)
conn.commit()

pd.read_sql_query("SELECT * FROM matches;", conn)

,match_id,home_team_id,away_team_id,stadium_id
0,1,1,2,1


## Summary
- Built a 4-table relational database (`teams`, `players`, `stadiums`, `matches`) from scratch
- Practiced `CREATE TABLE`, `PRIMARY KEY`, `FOREIGN KEY`, `NOT NULL`, `INSERT INTO` (single + multi-row)
- Learned that `PRAGMA foreign_keys = ON;` is required for SQLite to actually enforce relationships
- Confirmed the rule: **insert parent table data before child table data**
- Next: `SELECT`, `WHERE`, filtering real data across these linked tables (Day 37)